# 🔬 Spectrum-SLM — `.pth` File Explorer
### Every dataset and checkpoint file viewed as a CSV-style DataFrame

**How to use:** Run all cells top-to-bottom (`Kernel → Restart and Run All`).

For each `.pth` file you will see:
- 📋 **Column schema** — exactly like a CSV header
- 🗃️ **`df.head(5)`** — first 5 rows as a spreadsheet
- 📐 **`df.dtypes`** — data type of each column
- 📊 **`df.describe()`** — min / max / mean / std statistics


In [1]:
import os, sys, torch, numpy as np, pandas as pd, warnings, pickle
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

ROOT = os.path.dirname(os.path.abspath("__file__"))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

SU   = os.path.join(ROOT, "Secondary_User")      # dataset .pth files
CKPT = os.path.join(ROOT, "checkpoints")         # checkpoint .pt files
SLM  = os.path.join(ROOT, "slm_checkpoints")     # legacy checkpoint copies

pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", "{:.4f}".format)
print("✔  Imports OK")
print("   ROOT :", ROOT)


✔  Imports OK
   ROOT : c:\Users\ASUS Vivo book\Desktop\Complete-Data-Science-With-Machine-Learning-And-NLP-2024-main\SDR_Data


---
## 🛠️ Helper — convert any `.pth` to a pandas DataFrame

In [2]:
def binned_pth_to_df(path, modulation_name="unknown"):
    """
    Load a psd_binned_by_snr_*.pth and return a tidy DataFrame.

    CSV column layout
    -----------------
    snr_bin   | pu_label | modulation | bin_000 | bin_001 | … | bin_175
    float32   | int64    | str        | float32 | float32 |   | float32

    Total columns : 179  (snr_bin + pu_label + modulation + 176 PSD bins)
    One row       : one captured PSD snapshot
    """
    try:
        data  = torch.load(path, map_location="cpu", weights_only=False)
    except Exception as e:
        print(f"Error loading {path}: {e}")
        cols = ["snr_bin", "pu_label", "modulation"] + [f"bin_{i:03d}" for i in range(176)]
        return pd.DataFrame(columns=cols)
    
    bins  = data["bins"]           # [4.0, 6.0, …, 20.0]
    pairs = data["pairs_by_bin"]

    records = []
    for snr_val in bins:
        for psd_vec, pu_lbl in pairs.get(snr_val, []):
            psd = np.array(psd_vec, dtype=np.float32).ravel()
            psd = psd[:176] if len(psd) >= 176 else np.pad(psd, (0, 176 - len(psd)))
            row = [float(snr_val), int(pu_lbl), modulation_name] + psd.tolist()
            records.append(row)

    cols = ["snr_bin", "pu_label", "modulation"] + [f"bin_{i:03d}" for i in range(176)]
    return pd.DataFrame(records, columns=cols)


def log_pth_to_df(path, modulation_name="unknown", default_snr=10.0):
    """
    Load a psd_log_*.pth and return a tidy DataFrame.

    CSV column layout
    -----------------
    snr_db  | pu_label | modulation | bin_000 | bin_001 | … | bin_175
    float32 | int64    | str        | float32 | float32 |   | float32

    Total columns : 179
    """
    try:
        data = torch.load(path, map_location="cpu", weights_only=False)
    except Exception as e:
        print(f"Error loading {path}: {e}")
        cols = ["snr_db", "pu_label", "modulation"] + [f"bin_{i:03d}" for i in range(176)]
        return pd.DataFrame(columns=cols)

    if isinstance(data, torch.Tensor):
        psd_arr = data.numpy().astype(np.float32)          # (N, 176)
        pu_arr  = np.ones(len(psd_arr), dtype=int)
        snr_arr = np.full(len(psd_arr), default_snr)
    elif isinstance(data, dict):
        psd_arr = data.get("psd", data.get("data", None))
        if isinstance(psd_arr, torch.Tensor):
            psd_arr = psd_arr.numpy().astype(np.float32)
        pu_arr  = np.array(data.get("label", np.ones(len(psd_arr), dtype=int)), dtype=int)
        snr_arr = np.array(data.get("snr",   np.full(len(psd_arr), default_snr)))
    else:
        raise ValueError(f"Unknown log-pth format: {type(data)}")

    df = pd.DataFrame(psd_arr, columns=[f"bin_{i:03d}" for i in range(176)])
    df.insert(0, "modulation", modulation_name)
    df.insert(0, "pu_label",   pu_arr)
    df.insert(0, "snr_db",     snr_arr.astype(np.float32))
    return df


def checkpoint_to_df(path):
    """
    Load a model checkpoint .pt and return a DataFrame of the state_dict.

    CSV column layout (one row = one weight tensor / layer)
    -------------------------------------------------------
    layer_name | shape | num_params | dtype | mean | std | min | max
    str        | str   | int        | str   | f32  | f32 | f32 | f32
    """
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    sd   = ckpt["model"] if isinstance(ckpt, dict) and "model" in ckpt else ckpt

    rows = []
    for name, tensor in sd.items():
        t = tensor.float()
        rows.append({
            "layer_name" : name,
            "shape"      : str(tuple(tensor.shape)),
            "num_params" : tensor.numel(),
            "dtype"      : str(tensor.dtype).replace("torch.", ""),
            "mean"       : round(t.mean().item(), 5),
            "std"        : round(t.std().item(),  5),
            "min"        : round(t.min().item(),  5),
            "max"        : round(t.max().item(),  5),
        })
    return pd.DataFrame(rows), ckpt


print("✔  Helpers ready")


✔  Helpers ready


---
## 📦 File 1 — `psd_binned_by_snr_bpsk.pth` (BPSK · 90 MB)

**What it stores:** BPSK radio captures at 9 SNR levels (4–20 dB).  
BPSK = 1 bit/symbol → narrowest, single-lobe PSD spectrum.

| Layer | Detail |
|---|---|
| Format | Binned dict |
| Modulation | BPSK |
| Columns | 179 (snr_bin + pu_label + modulation + bin_000…bin_175) |
| Approx. rows | ~380,000 |


In [3]:
fpath = os.path.join(SU, "psd_binned_by_snr_bpsk.pth")
if os.path.exists(fpath):
    df_bpsk = binned_pth_to_df(fpath, "BPSK")

    print(f"Shape  : {df_bpsk.shape}   →  ({df_bpsk.shape[0]:,} rows  x  {df_bpsk.shape[1]} columns)")
    print(f"\n── Samples per SNR bin ────────────────────────")
    print(df_bpsk.groupby("snr_bin").size().rename("count").to_string())

    print(f"\n── Column dtypes ──────────────────────────────")
    print(df_bpsk.dtypes.to_string())

    print(f"\n── First 5 rows (label columns + first 4 PSD bins) ──")
    display_cols = ["snr_bin", "pu_label", "modulation", "bin_000", "bin_001", "bin_002", "bin_003"]
    print(df_bpsk[display_cols].head())

    print(f"\n── PSD bin statistics (bin_000 … bin_175) ─────")
    print(df_bpsk.iloc[:, 3:].describe().round(3).to_string())

    print(f"\n── PU label distribution ──────────────────────")
    print(df_bpsk["pu_label"].value_counts().rename("count"))
else:
    print("[SKIP] File not found:", fpath)


Shape  : (37895, 179)   →  (37,895 rows  x  179 columns)

── Samples per SNR bin ────────────────────────
snr_bin
4.0000        14
6.0000      4063
8.0000      8922
10.0000     8082
12.0000    14178
14.0000     2378
16.0000      247
18.0000       10
20.0000        1

── Column dtypes ──────────────────────────────
snr_bin       float64
pu_label        int64
modulation     object
bin_000       float64
bin_001       float64
bin_002       float64
bin_003       float64
bin_004       float64
bin_005       float64
bin_006       float64
bin_007       float64
bin_008       float64
bin_009       float64
bin_010       float64
bin_011       float64
bin_012       float64
bin_013       float64
bin_014       float64
bin_015       float64
bin_016       float64
bin_017       float64
bin_018       float64
bin_019       float64
bin_020       float64
bin_021       float64
bin_022       float64
bin_023       float64
bin_024       float64
bin_025       float64
bin_026       float64
bin_027       float64
bi

---
## 📦 File 2 — `psd_binned_by_snr_qpsk.pth` (QPSK · 63 MB)

**What it stores:** QPSK captures · 2 bits/symbol · wider lobe than BPSK.  
Same 178-column layout as the BPSK file.


In [4]:
fpath = os.path.join(SU, "psd_binned_by_snr_qpsk.pth")
if os.path.exists(fpath):
    df_qpsk = binned_pth_to_df(fpath, "QPSK")
    print(f"Shape  : {df_qpsk.shape[0]:,} rows  x  {df_qpsk.shape[1]} columns")

    display_cols = ["snr_bin", "pu_label", "modulation", "bin_000", "bin_001", "bin_002", "bin_003"]
    print("\n── First 5 rows ────────────────────────────────")
    print(df_qpsk[display_cols].head())

    print("\n── PSD summary stats ───────────────────────────")
    print(df_qpsk.iloc[:, 3:].describe().round(3).loc[["mean","std","min","max"]].to_string())
else:
    print("[SKIP]", fpath)


Shape  : 26,455 rows  x  179 columns

── First 5 rows ────────────────────────────────
   snr_bin  pu_label modulation  bin_000  bin_001  bin_002  bin_003
0   4.0000         0       QPSK -25.3737 -31.3613 -23.6676 -20.0466
1   4.0000         0       QPSK -40.6395 -34.1049 -31.2550 -27.7740
2   4.0000         0       QPSK -15.9927 -14.5112 -14.4238 -13.3268
3   4.0000         0       QPSK -32.8560 -30.5889 -28.6440 -27.6272
4   4.0000         0       QPSK -16.7452 -17.5282 -18.0946 -18.2129

── PSD summary stats ───────────────────────────
      bin_000  bin_001  bin_002  bin_003  bin_004  bin_005  bin_006  bin_007  bin_008  bin_009  bin_010  bin_011  bin_012  bin_013  bin_014  bin_015  bin_016  bin_017  bin_018  bin_019  bin_020  bin_021  bin_022  bin_023  bin_024  bin_025  bin_026  bin_027  bin_028  bin_029  bin_030  bin_031  bin_032  bin_033  bin_034  bin_035  bin_036  bin_037  bin_038  bin_039  bin_040  bin_041  bin_042  bin_043  bin_044  bin_045  bin_046  bin_047   bin_048  bin_049